# 1 · Talking to a model on NRP

*Adapted from Daniel Diaz's CLARIPHY lesson [Chat with LLMs on NRP](https://training.nrp-nautilus.io/clariphy/2_chat.html) ([nrp-nautilus/nrp-training](https://github.com/nrp-nautilus/nrp-training/tree/materials/clariphy)).*

This notebook calls NRP's managed LLMs from Python, through the OpenAI-compatible endpoint at
`https://ellm.nrp-nautilus.io/v1`. The models run on NRP's shared inference cluster, so no GPU is
needed here. We use the two models of the hackathon, **`deepseek-v4-flash`** and **`glm-5`**.

**Run cells with Shift+Enter, top to bottom.**

| Step | Topic |
|---|---|
| 1 | Setup check |
| 2 | Basic chat & the `chat()` helper |
| 3 | What the model actually receives |
| 4 | System prompts |
| 5 | Multi-turn chat |

## Available Models

Quick reference — the same catalog from [Lesson 1](https://training.nrp-nautilus.io/clariphy/1_intro.html#1-managed-llm-service), NRP model ID alongside the underlying HuggingFace model:

| Model | HuggingFace ID | Parameters | Context | Tools | Vision | Notes |
|---|---|---|---|---|---|---|
| `qwen3` | `Qwen/Qwen3.5-397B-A17B-FP8` | 397B (17B active MoE) | 1.01M | ✓ | image, video | Largest context |
| `qwen3-small` | `Qwen/Qwen3.6-27B` | 27B | 1.01M | ✓ | image, video | |
| `gpt-oss` | `openai/gpt-oss-120b` | 120B | 131K | ✓ | — | Strong at code |
| `gemma` | `google/gemma-4-31B-it-qat-w4a16-ct` | 31B | 262K | ✓ | image, video | |
| `gemma-small` | `google/gemma-4-12B-it-qat-w4a16-ct` | 12B | 262K | ✓ | image, video | *Evaluating* — fast, good default |
| `minimax-m2` | `MiniMaxAI/MiniMax-M2.7` | 230B | 204K | ✓ | — | *Evaluating* — strong reasoning |
| `glm-5` | `nvidia/GLM-5.2-NVFP4` | 744B | 300K | ✓ | — | *Evaluating* |
| `deepseek-v4-flash` | `deepseek-ai/DeepSeek-V4-Flash-0731` | 304B | 1.05M | ✓ | — | *Evaluating* |
| `kimi` | `moonshotai/Kimi-K2.7-Code` | 1T MoE | 131K | ✓ | image, video | *Evaluating* |
| `qwen3-embedding` | `Qwen/Qwen3-VL-Embedding-8B` | 8B | 262K | — | image, video | Embeddings only — semantic search/RAG |

*Evaluating* models are under active testing — configuration can change without notice. This notebook uses `deepseek-v4-flash` and `glm-5`, the models of the hackathon; any other row works the same way. Check the [live list](https://nrp.ai/documentation/userdocs/ai/llm-managed/models/) or the [LLM status dashboard](https://nrp.ai/llm-status/) if something isn't responding.

## 1. Setup Check

Verify your token and the OpenAI client.

In [ ]:
%pip install -q openai   # NEW: the workshop's Binder image has no OpenAI client; elsewhere a no-op

Run the cell below and paste your **LLM API key** (from the training link) into the box that appears, then press
Enter. *NEW in this version:* the key is saved to a private file, `~/.nrp-llm-token`, readable only by you;
notebooks 2 and 3 and opencode read it from there, so you paste it **once per session** and it never sits in a
notebook. A Binder restart wipes the file: just run this cell again.

In [ ]:
import os, getpass
from pathlib import Path

KEY_FILE = Path.home() / ".nrp-llm-token"          # private: readable only by you
if not KEY_FILE.exists():
    KEY_FILE.write_text(getpass.getpass("Paste your NRP LLM API key, then press Enter: ").strip())
    KEY_FILE.chmod(0o600)
    print("Saved to", KEY_FILE)

# Set both explicitly, replacing anything already in the environment.
os.environ["OPENAI_API_KEY"] = KEY_FILE.read_text().strip()
os.environ["OPENAI_API_BASE"] = "https://ellm.nrp-nautilus.io/v1"
print("Key loaded from", KEY_FILE)

Listing the models shows the endpoint is reachable, but NRP answers that request even without a valid token — so the check also sends one tiny request to each model this notebook uses.

In [ ]:
import os
from openai import OpenAI

print("OPENAI_API_BASE =", os.environ.get("OPENAI_API_BASE", "NOT SET"))
key = os.environ.get("OPENAI_API_KEY", "")
print("OPENAI_API_KEY  =", key[:8] + "..." if key else "NOT SET")

client = OpenAI(
    api_key=os.environ["OPENAI_API_KEY"],
    base_url=os.environ["OPENAI_API_BASE"],
)

models = client.models.list()
print(f"\n{len(models.data)} models available:")
for m in sorted(models.data, key=lambda x: x.id):
    print(f"  {m.id}")

# Listing models doesn't check your token, so send one tiny request per model we use.
print()
for model in ["deepseek-v4-flash", "glm-5"]:
    try:
        client.chat.completions.create(
            model=model, max_tokens=1, messages=[{"role": "user", "content": "hi"}])
        print(f"  ✅ {model}")
    except Exception as e:
        print(f"  ❌ {model}: {type(e).__name__} {getattr(e, 'status_code', '')}")

If you see **`PermissionDeniedError 403`**: when *every* model fails, NRP is rejecting your token — check that you pasted your own key from the training-link page: delete the saved one with `!rm ~/.nrp-llm-token`, rerun the key cell, then this one, and that your account is in a namespace with LLM access (opening the training link adds you to it). When only one model fails, that model is unavailable to you right now — use another from the table above, e.g. `model="gpt-oss"`, wherever that model appears.

## Optional: Set Up Jupyter AI in JupyterLab

Everything below can also be done without writing any code, using
JupyterLab's built-in **Jupyter AI** extension — a chat sidebar backed by the
same NRP models.

1. Click the chat-bubble icon in the left sidebar to open **Jupyter AI
   Chat**, then click **Start Here** (or the gear icon) to open its settings.

   ![Jupyter AI welcome panel — click Start Here to open settings](../images/jupyter-ai.png)

2. Under **Language model**, point it at NRP instead of OpenAI's own API:
   - **Completion model**: choose `OpenAI (general interface)...` — this
     tells Jupyter AI to speak the OpenAI API format but let you supply your
     own server, instead of assuming platform.openai.com.
   - **Model ID**: the NRP model you want, e.g. `deepseek-v4-flash`.
   - **Base API URL (optional)**: `https://ellm.nrp-nautilus.io/v1` — the
     same `OPENAI_API_BASE` used everywhere else in this training.
   - Leave **Organization** and **Proxy** blank.
   - Under **API Keys**, paste your token into **OPENAI_API_KEY** — your
     LLM API key from [the training link](../README.md#before-you-start).

   ![Jupyter AI settings — Completion model, Model ID, Base API URL, and API key](../images/jupyter-ai2.png)

3. Close the settings panel and start chatting in the sidebar.

This is entirely optional — the rest of this notebook talks to NRP directly
through the `openai` Python package, which works the same everywhere
(JupyterLab, a script, your own machine) and doesn't depend on this
extension being installed.

## 2. Basic Chat & the `chat()` Helper

The OpenAI chat API takes a list of `messages`, each with a `role` and
`content`. You'll use three roles:

| Role | Purpose |
|---|---|
| `system` | The **system prompt** — instructions that set the model's behavior/persona for the whole conversation. Sent once, usually first. |
| `user` | What the human is asking. |
| `assistant` | The model's own previous replies — sent back on later turns so it remembers the conversation (see [Section 5](#5-multi-turn-interactive-chat)). |

A few parameters worth knowing, including the ones our `chat()` helper below exposes as Python arguments:

| Parameter | What it does |
|---|---|
| `model` | Which model to use — see the [model table](https://training.nrp-nautilus.io/clariphy/1_intro.html#1-managed-llm-service) in the intro lesson. |
| `messages` | The list of `{role, content}` turns described above. |
| `system` (`chat()` helper arg) | Your system prompt as a plain string. `chat()` wraps it into a `{"role": "system", "content": ...}` message for you, so you don't have to build the `messages` list by hand. |
| `max_tokens` | Hard cap on reply length. Reasoning models (`deepseek-v4-flash`, `glm-5`, `gpt-oss`, `qwen3`) spend part of this budget thinking privately before answering, so give them more room (1000+) or you may get an empty reply. |
| `temperature` | Randomness, from `0` (deterministic — same input gives the same answer) to `~1.5` (more varied/creative). `0.2` is a good default for factual or code answers. |

Define a reusable helper that handles both regular and reasoning models.
Reasoning models (`deepseek-v4-flash`, `glm-5`, `gpt-oss`, `qwen3`) think privately before
answering, so they need a larger `max_tokens`.

In [ ]:
def chat(prompt, model="deepseek-v4-flash", system=None, max_tokens=1200):
    msgs = []
    if system:
        msgs.append({"role": "system", "content": system})
    msgs.append({"role": "user", "content": prompt})
    resp = client.chat.completions.create(
        model=model, messages=msgs, max_tokens=max_tokens, temperature=0.2,
    )
    msg = resp.choices[0].message
    if msg.content:
        return msg.content
    # Reasoning models stream thinking into a separate field
    reasoning = getattr(msg, "reasoning", None) or getattr(msg, "reasoning_content", None)
    if reasoning:
        return f"(reasoning only — increase max_tokens):\n{reasoning}"
    return "(no content returned)"

In [ ]:
print(chat(
    "What is the CMS detector and what is it used for?",
    system="Answer in two sentences for an audience of physics graduate students.",
))

In [ ]:
# Code generation — the same helper, the other model
print(chat(
    "Write a short Python snippet using uproot to open a ROOT file called 'data.root',"
    " read the TTree named 'Events', and print the number of entries.",
    model="glm-5",
))

In [ ]:
# NEW: one question, both models, side by side
QUESTION = ("Explain why the invariant mass of two muons is a useful observable for"
            " searching for new particles decaying to mu+mu-. Three sentences.")
for model in ["deepseek-v4-flash", "glm-5"]:
    print(f"=== {model} ===")
    print(chat(QUESTION, model=model, max_tokens=2000), "\n")

## 3. What the model actually receives

*NEW in this version.* A chat model has no memory and no view of your notebook. Each request carries
everything the model will know about the conversation: the `messages` list. The server turns that list
into one long text using the model's **chat template** (role markers, special tokens, sometimes default
instructions), and bills you in **tokens**, roughly ¾ of a word each.

Here is one request exactly as it goes over the wire, and the bill that comes back:

In [ ]:
import json

request = {
    "model": "deepseek-v4-flash",
    "messages": [
        {"role": "system", "content": "Answer in one sentence."},
        {"role": "user", "content": "What is the CMS detector?"},
    ],
    "max_tokens": 1200,
}
print(json.dumps(request, indent=2))     # this JSON is the entire request body

resp = client.chat.completions.create(**request)
print("\nreply:", resp.choices[0].message.content)
print("tokens in (prompt):", resp.usage.prompt_tokens,
      "| tokens out (completion):", resp.usage.completion_tokens)

Two short messages, yet the prompt is billed at more tokens than the words you wrote. Send the same request
to both models and compare:

In [ ]:
for model in ["deepseek-v4-flash", "glm-5"]:
    r = client.chat.completions.create(**{**request, "model": model})
    print(f"{model:18s} prompt tokens: {r.usage.prompt_tokens:4d}   completion tokens: {r.usage.completion_tokens}")

Where do the extra tokens come from? Send almost nothing — `"hi"` — and then ten of them:

In [ ]:
for model in ["deepseek-v4-flash", "glm-5"]:
    for text in ["hi", "hi " * 10]:
        r = client.chat.completions.create(model=model, max_tokens=5,
                                           messages=[{"role": "user", "content": text}])
        print(f"{model:18s} {text!r:36s} prompt tokens: {r.usage.prompt_tokens}")

Ten more words cost about ten more tokens on both models. But even a bare `"hi"` cost ~97 prompt tokens on
DeepSeek and ~13 on GLM when we tried (11 Oct 2026): every request carries text you never wrote, added around
your messages by the server (the **chat template**: role markers, special tokens, and for some models a default
preamble). You never see it, but the model reads it and you pay for it.

The completion count includes the model's private **reasoning**: both models think before they answer,
and that thinking is billed and counted against `max_tokens` too (`msg.reasoning_content` holds it).

### The conversation is resent every turn

To continue a conversation you send the **whole history again**: the model re-reads everything on every
turn. That history is the model's **context**, and each model has a maximum, its context window. An agent's
context is its system prompt, its tool descriptions and every tool call and result so far; that is what
fills up during a long run.

In [ ]:
history = [{"role": "system", "content": "You are a concise physics tutor. Two sentences per answer."}]
for q in ["What is a muon?", "How heavy is it compared with the electron?", "And what does it decay to?"]:
    history.append({"role": "user", "content": q})
    r = client.chat.completions.create(model="deepseek-v4-flash", messages=history, max_tokens=1500)
    answer = r.choices[0].message.content
    history.append({"role": "assistant", "content": answer})
    print(f"Q: {q}   [{r.usage.prompt_tokens} prompt tokens sent]\nA: {answer}\n")

The prompt grows every turn. Now ask the last question **without** the history: the model has no idea
what "it" is, because nothing outside the `messages` list exists for it.

In [ ]:
print(chat("And what does it decay to?", system="You are a concise physics tutor. Two sentences per answer."))

## 4. System Prompts & Personas

The system prompt defines the model's role. Below: the same CMS question answered
by four different roles. Swap in whatever is most useful for your workflow.

In [ ]:
QUESTION = "How do I apply a muon pT > 20 GeV selection in CMS NanoAOD with Python?"

ROLES = {
    "Teaching assistant": (
        "You are a supportive teaching assistant for CMS physicists. Explain "
        "clearly with short examples, guiding the learner toward the answer."),
    "Technical coder": (
        "You are an expert HEP software engineer. Write clean, correct Python "
        "using coffea or uproot, then briefly explain it and note edge cases."),
    "Concise expert": (
        "You are a senior CMS physicist. Answer precisely in a few sentences "
        "for a graduate-level audience. No filler."),
    "Documentation writer": (
        "You are a CMS documentation writer. Structure your answer with headings, "
        "a code block, and a note on common pitfalls."),
}

for role, system in ROLES.items():
    print(f"\n{'='*60}\n=== {role} ===")
    print(chat(QUESTION, system=system))

## 5. Multi-Turn Interactive Chat

Run this cell, then type questions at the prompt. The model remembers context
across turns — like office hours. Type `quit` to stop, `reset` to clear history.

In [ ]:
ROLE  = "Teaching assistant"   # change to any key in ROLES above
MODEL = "deepseek-v4-flash"   # or glm-5, gpt-oss, qwen3

SYSTEMS = {
    "Teaching assistant": (
        "You are a supportive teaching assistant for CMS physicists. Explain "
        "clearly, build intuition with examples, and guide the learner."),
    "CMS expert": (
        "You are an expert CMS physicist. Answer precisely and technically."),
    "Technical coder": (
        "You are an expert HEP software engineer. Write clean Python and explain it briefly."),
}

history = []
print(f"Chatting as: {ROLE} ({MODEL}).  Type 'quit' to stop, 'reset' to clear.\n")
while True:
    try:
        msg = input("You: ").strip()
    except (EOFError, KeyboardInterrupt):
        print("\n(ended)"); break
    if msg.lower() in ("quit", "exit", "q", ""):
        print("Bye!"); break
    if msg.lower() == "reset":
        history.clear(); print("(conversation cleared)\n"); continue
    history.append({"role": "user", "content": msg})
    r = client.chat.completions.create(
        model=MODEL, max_tokens=1000, temperature=0.5,
        messages=[{"role": "system", "content": SYSTEMS[ROLE]}] + history)
    reply = r.choices[0].message.content or "(no reply)"
    history.append({"role": "assistant", "content": reply})
    print(f"AI: {reply}\n   ({r.usage.prompt_tokens} prompt tokens sent this turn)\n")

## Takeaways

- One request = one `messages` list. The model sees nothing else: no files, no memory, no notebook.
- The **system prompt** is just the first message; it shapes every answer that follows.
- **Context** is the whole history, resent on every turn and billed in tokens; it is finite.
- Models wrap the same messages differently and reason before answering: compare `deepseek-v4-flash` and `glm-5`.

**Next:** [2 · An agent, by hand](2_agent_by_hand.ipynb), where the model starts asking *you* to run things.